In [2]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-12-11 04:26:20--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8003::154, 2606:50c0:8000::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.4’

input.txt.4         100%[===================>]   1.06M  --.-KB/s    in 0.07s   

2025-12-11 04:26:20 (15.8 MB/s) - ‘input.txt.4’ saved [1115394/1115394]



In [1]:
with open('input.txt', 'r') as f:
    text = f.read()

In [2]:
print("Length of the dataset: ", len(text))

Length of the dataset:  1115394


In [3]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [4]:
for char in "\n!$&',-.3:;?":
    if char == '\n':
        text = text.replace(char, ' ')
    else:
        text = text.replace(char, '')

print(text[:1000])

First Citizen Before we proceed any further hear me speak  All Speak speak  First Citizen You are all resolved rather to die than to famish  All Resolved resolved  First Citizen First you know Caius Marcius is chief enemy to the people  All We knowt we knowt  First Citizen Let us kill him and well have corn at our own price Ist a verdict  All No more talking ont let it be done away away  Second Citizen One word good citizens  First Citizen We are accounted poor citizens the patricians good What authority surfeits on would relieve us if they would yield us but the superfluity while it were wholesome we might guess they relieved us humanely but they think we are too dear the leanness that afflicts us the object of our misery is as an inventory to particularise their abundance our sufferance is a gain to them Let us revenge this with our pikes ere we become rakes for the gods know I speak this in hunger for bread not in thirst for revenge  Second Citizen Would you proceed especially again

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

 ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
53


In [6]:
stoi = { ch:i for i,ch in enumerate(chars)}
itos = { i:ch for i,ch in enumerate(chars)}
for ch in chars:
    assert itos[stoi[ch]] == ch
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
assert decode(encode(text[100-130])) == text[100-130]

In [7]:
import torch
data = torch.tensor(encode(text))

In [8]:
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1060970]) torch.int64
tensor([ 6, 35, 44, 45, 46,  0,  3, 35, 46, 35, 52, 31, 40,  0,  2, 31, 32, 41,
        44, 31,  0, 49, 31,  0, 42, 44, 41, 29, 31, 31, 30,  0, 27, 40, 51,  0,
        32, 47, 44, 46, 34, 31, 44,  0, 34, 31, 27, 44,  0, 39, 31,  0, 45, 42,
        31, 27, 37,  0,  0,  1, 38, 38,  0, 19, 42, 31, 27, 37,  0, 45, 42, 31,
        27, 37,  0,  0,  6, 35, 44, 45, 46,  0,  3, 35, 46, 35, 52, 31, 40,  0,
        25, 41, 47,  0, 27, 44, 31,  0, 27, 38, 38,  0, 44, 31, 45, 41, 38, 48,
        31, 30,  0, 44, 27, 46, 34, 31, 44,  0, 46, 41,  0, 30, 35, 31,  0, 46,
        34, 27, 40,  0, 46, 41,  0, 32, 27, 39, 35, 45, 34,  0,  0,  1, 38, 38,
         0, 18, 31, 45, 41, 38, 48, 31, 30,  0, 44, 31, 45, 41, 38, 48, 31, 30,
         0,  0,  6, 35, 44, 45, 46,  0,  3, 35, 46, 35, 52, 31, 40,  0,  6, 35,
        44, 45, 46,  0, 51, 41, 47,  0, 37, 40, 41, 49,  0,  3, 27, 35, 47, 45,
         0, 13, 27, 44, 29, 35, 47, 45,  0, 35, 45,  0, 29, 34, 35, 31, 32,  0,
      

In [9]:
# Split train and test data
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [10]:
block_size = 8
train_data[:block_size + 1]

tensor([ 6, 35, 44, 45, 46,  0,  3, 35, 46])

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"When input is {context} then target: {target}")

When input is tensor([6]) then target: 35
When input is tensor([ 6, 35]) then target: 44
When input is tensor([ 6, 35, 44]) then target: 45
When input is tensor([ 6, 35, 44, 45]) then target: 46
When input is tensor([ 6, 35, 44, 45, 46]) then target: 0
When input is tensor([ 6, 35, 44, 45, 46,  0]) then target: 3
When input is tensor([ 6, 35, 44, 45, 46,  0,  3]) then target: 35
When input is tensor([ 6, 35, 44, 45, 46,  0,  3, 35]) then target: 46


In [12]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will be processed in parallel?
block_size = 8 # what is the maximum context length for prediction
import random
def get_example(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    i = random.randint(0, len(data) - block_size)
    x = data[i: i + block_size]
    y = data[i + 1: i + block_size + 1]
    return x, y

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i: i + block_size] for i in ix])
    y = torch.stack([data[i + 1: i + block_size + 1] for i in ix])
    return x, y

In [13]:
xb, yb = get_batch('train')

In [18]:
import torch
from torch import nn
from torch.nn import functional as F
torch.manual_seed(1337)
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_size = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_size(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # become (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T + 1)
        return idx

In [15]:

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb)

print(logits)

tensor([[[ 0.3854, -0.8898,  1.3468,  ..., -1.7526, -1.1510,  0.8768],
         [-0.1310, -1.7969, -0.5603,  ..., -0.8036, -1.3313,  1.9921],
         [ 2.1139, -2.1624, -0.7994,  ..., -0.1536,  1.9895, -0.1837],
         ...,
         [ 0.3886,  1.0050, -1.2381,  ...,  0.8741,  0.1697, -2.4297],
         [-1.1513,  0.1186, -0.3123,  ..., -0.7202,  0.8811, -0.2593],
         [-0.1310, -1.7969, -0.5603,  ..., -0.8036, -1.3313,  1.9921]],

        [[-0.3366,  0.3025, -0.4242,  ..., -0.0157,  1.7251, -1.9974],
         [ 0.1808, -0.0700, -0.3596,  ...,  1.3514, -0.2759, -1.5108],
         [-0.3366,  0.3025, -0.4242,  ..., -0.0157,  1.7251, -1.9974],
         ...,
         [-0.0815, -0.3447,  0.8249,  ...,  1.2207,  0.0814, -0.8279],
         [-1.8638, -1.0699, -0.6119,  ..., -1.2395,  1.0547,  0.5108],
         [ 0.1808, -0.0700, -0.3596,  ...,  1.3514, -0.2759, -1.5108]],

        [[ 1.1956, -0.6034,  1.7504,  ..., -1.0606,  0.6451, -1.9633],
         [-0.3366,  0.3025, -0.4242,  ..., -0

In [16]:
m = BigramLanguageModel(vocab_size)
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [17]:
for r in range(100000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    print(r, loss.item())
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

0 4.63077449798584
1 4.777806758880615
2 4.4031982421875
3 4.709268093109131
4 4.2898454666137695
5 4.5346784591674805
6 4.550993919372559
7 4.3985185623168945
8 4.216459274291992
9 4.202322006225586
10 4.447339057922363
11 4.476679801940918
12 4.521987438201904
13 4.561304092407227
14 4.198914527893066
15 4.368988990783691
16 4.343623638153076
17 4.522928714752197
18 4.277977466583252
19 4.307976245880127
20 4.391712665557861
21 4.388931751251221
22 4.646235466003418
23 4.710846900939941
24 4.563187599182129
25 4.40189790725708
26 4.502045631408691
27 4.633767127990723
28 4.589503765106201
29 4.192300796508789
30 4.308565616607666
31 4.2945685386657715
32 4.441539287567139
33 4.222686290740967
34 4.290709018707275
35 3.9966602325439453
36 4.758399963378906
37 4.4936909675598145
38 4.853877544403076
39 4.360931396484375
40 4.211662292480469
41 4.313292026519775
42 4.584230422973633
43 4.093976974487305
44 4.572533130645752
45 4.135244369506836
46 4.3520307540893555
47 4.093037128448486

KeyboardInterrupt: 

In [22]:
decode(m.generate(torch.tensor(encode(' A')).unsqueeze(0), 30)[0].tolist())

' AQfSAMgr VvtDwnjSzxwdWuzPcMaclR'

In [144]:
nn.Parameter(torch.randn(3, 4, 2))

Parameter containing:
tensor([[[-0.5433, -1.2852],
         [-0.1862, -1.2648],
         [ 0.6976,  0.6512],
         [ 0.9831,  1.3604]],

        [[-1.4891,  0.2735],
         [ 2.7942,  0.2806],
         [-0.7959, -0.7341],
         [ 1.0291, -1.3070]],

        [[ 0.9760, -1.1851],
         [-0.7724, -1.5755],
         [-0.5857,  0.6632],
         [-0.2597,  3.1973]]], requires_grad=True)

In [23]:
class TrigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.trigram = nn.Parameter(torch.randn(vocab_size, vocab_size, vocab_size))

    def forward(self, idx, targets=None):
        start_tokens = torch.full((idx.shape[0], 1), stoi[' '], dtype=idx.dtype, device=idx.device)
        prev_tokens = torch.cat([start_tokens, idx[:, :-1]], dim=1)
        bi_idx = torch.stack([prev_tokens, idx], dim=2)  # Shape: [batch, seq_len, 2]
        # idx and targets are both (B, T) tensor of integers
        logits = self.trigram[bi_idx[..., 0 ], bi_idx[..., 1]]

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # become (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T + 1)
        return idx

In [24]:
xb, yb = get_batch('train')

In [25]:
tri = TrigramLanguageModel(vocab_size)

In [26]:
tri_optimizer = torch.optim.AdamW(tri.parameters(), lr=1e-4)

In [238]:
for r in range(100000):
    xb, yb = get_batch('train')
    logits, loss = tri(xb, yb)
    print(r, loss.item())
    tri_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    tri_optimizer.step()

0 2.1543474197387695
1 2.1129791736602783
2 2.3255133628845215
3 2.488454580307007
4 1.8129286766052246
5 2.3671159744262695
6 1.8495092391967773
7 1.9197163581848145
8 2.0214741230010986
9 2.115936517715454
10 1.8426899909973145
11 2.096170663833618
12 1.7754589319229126
13 2.5880308151245117
14 1.8891583681106567
15 1.9046634435653687
16 2.2052853107452393
17 1.7761017084121704
18 1.8182244300842285
19 2.1429848670959473
20 1.8630566596984863
21 1.9349873065948486
22 2.08461856842041
23 2.1495795249938965
24 1.7765536308288574
25 1.7985002994537354
26 2.0298566818237305
27 2.0970325469970703
28 1.9359582662582397
29 1.8448305130004883
30 2.227771520614624
31 1.91592538356781
32 2.0632922649383545
33 2.269514799118042
34 1.9418065547943115
35 2.18406343460083
36 2.6424012184143066
37 1.8474916219711304
38 2.0484118461608887
39 1.7113516330718994
40 2.0115418434143066
41 2.0975401401519775
42 1.7053947448730469
43 1.90841543674469
44 1.818461537361145
45 1.7561088800430298
46 2.1524622

In [29]:
decode(tri.generate(torch.tensor([encode('tr')]), 30)[0].tolist())

'trpcevvtfSLdcmNgh BLYyKIyvJZftyl'

In [30]:
tri(torch.tensor([[12]]))[0]

tensor([[[-0.9673,  0.6091,  0.2638, -1.3711,  0.2165,  1.3970, -0.4574,
           1.0669, -0.1515, -0.0376,  0.8555, -0.4845, -1.1641, -2.0859,
          -0.1966, -1.4722, -0.0385,  2.1131,  0.5180,  0.3778, -2.6795,
          -0.7124,  0.1483, -1.1713, -2.0710,  0.4737,  0.3819,  0.2300,
           0.7485,  0.6605,  2.9092, -1.1255, -0.7368,  0.3089,  0.7815,
          -0.2260,  1.3179, -0.1356,  0.9973,  0.1311, -2.8175, -0.6779,
          -0.0903,  1.7956,  0.0543, -2.2493, -0.1277,  1.3544, -0.8345,
           0.1617, -0.6207, -0.1478, -1.5253]]], grad_fn=<IndexBackward0>)

In [31]:
tri.trigram[0, 12]

tensor([-0.9673,  0.6091,  0.2638, -1.3711,  0.2165,  1.3970, -0.4574,  1.0669,
        -0.1515, -0.0376,  0.8555, -0.4845, -1.1641, -2.0859, -0.1966, -1.4722,
        -0.0385,  2.1131,  0.5180,  0.3778, -2.6795, -0.7124,  0.1483, -1.1713,
        -2.0710,  0.4737,  0.3819,  0.2300,  0.7485,  0.6605,  2.9092, -1.1255,
        -0.7368,  0.3089,  0.7815, -0.2260,  1.3179, -0.1356,  0.9973,  0.1311,
        -2.8175, -0.6779, -0.0903,  1.7956,  0.0543, -2.2493, -0.1277,  1.3544,
        -0.8345,  0.1617, -0.6207, -0.1478, -1.5253],
       grad_fn=<SelectBackward0>)

In [32]:
x1b = []
for x in xb:
    x1 = [[stoi[' '], x[0].item()]]
    for i in range(1, len(x)):
        x1.append([x[i - 1].item(), x[i].item()])
    x1b.append(x1)

print(torch.tensor(x1b)[0], yb[0])

tensor([[ 0, 31],
        [31, 30],
        [30,  0],
        [ 0, 42],
        [42, 44],
        [44, 41],
        [41, 36],
        [36, 31]]) tensor([30,  0, 42, 44, 41, 36, 31, 29])


In [33]:
start_tokens = torch.full((xb.shape[0], 1), stoi[' '], dtype=xb.dtype, device=xb.device)
prev_tokens = torch.cat([start_tokens, xb[:, :-1]], dim=1)
x1b_tensor = torch.stack([prev_tokens, xb], dim=2)  # Shape: [batch, seq_len, 2]

In [34]:
xb.shape

torch.Size([4, 8])

In [35]:
x1b

[[[0, 31], [31, 30], [30, 0], [0, 42], [42, 44], [44, 41], [41, 36], [36, 31]],
 [[0, 31], [31, 0], [0, 33], [33, 41], [41, 41], [41, 30], [30, 0], [0, 3]],
 [[0, 38], [38, 27], [27, 45], [45, 34], [34, 51], [51, 0], [0, 46], [46, 41]],
 [[0, 45], [45, 0], [0, 46], [46, 34], [34, 51], [51, 0], [0, 31], [31, 51]]]

In [36]:
r = nn.RNN(16, 32)

In [37]:
for n, p in r.named_parameters():
    print(n, p.shape)

weight_ih_l0 torch.Size([32, 16])
weight_hh_l0 torch.Size([32, 32])
bias_ih_l0 torch.Size([32])
bias_hh_l0 torch.Size([32])


In [38]:
(torch.randn(32, 1) * r.weight_ih_l0).shape

torch.Size([32, 16])

In [39]:
torch.randn(32, 16) @ torch.randn(16, 1)

tensor([[-2.0916],
        [ 3.5597],
        [-1.6569],
        [-1.0641],
        [-1.2043],
        [ 1.9808],
        [-3.3616],
        [-0.2971],
        [ 1.3289],
        [ 3.2967],
        [-3.5206],
        [ 2.2103],
        [ 0.8278],
        [-1.7742],
        [-0.2464],
        [-4.9155],
        [ 3.0570],
        [ 0.2650],
        [ 1.3153],
        [-1.2335],
        [ 1.9044],
        [-2.7101],
        [-1.4620],
        [-0.3139],
        [-0.6997],
        [ 0.2723],
        [ 4.4108],
        [-4.8988],
        [ 2.1245],
        [-0.4966],
        [ 2.1958],
        [ 0.8612]])

## The mathematical trick in self-attention

In [40]:
# consider the following toy example
torch.manual_seed(1337)
B, T, C = 4, 8, 2 # batch, time, channels
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [41]:
# We want x[b, t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C), dtype=x.dtype)
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, 0)

xbow[1]

tensor([[ 1.3488, -0.1396],
        [ 0.8173,  0.4127],
        [-0.1342,  0.4395],
        [ 0.2711,  0.4774],
        [ 0.2421,  0.0694],
        [ 0.0084,  0.0020],
        [ 0.0712, -0.1128],
        [ 0.2527,  0.2149]])

In [42]:
mask = torch.tril(torch.ones(T, T))
mask = mask / mask.sum(1, keepdim=True)
xbow2 = mask @ x
xbow2[1]

tensor([[ 1.3488, -0.1396],
        [ 0.8173,  0.4127],
        [-0.1342,  0.4395],
        [ 0.2711,  0.4774],
        [ 0.2421,  0.0694],
        [ 0.0084,  0.0020],
        [ 0.0712, -0.1128],
        [ 0.2527,  0.2149]])

In [43]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei[tril == 0] = float('-inf')
wei = F.softmax(wei, dim=1)
out = wei @ x
out[3]

tensor([[ 1.6455, -0.8030],
        [ 1.4985, -0.5395],
        [ 0.4954,  0.3420],
        [ 1.0623, -0.1802],
        [ 1.1401, -0.4462],
        [ 1.0870, -0.4071],
        [ 1.0430, -0.1299],
        [ 1.1138, -0.1641]])

In [44]:
torch.eq(out, xbow)

tensor([[[ True,  True],
         [ True,  True],
         [ True, False],
         [ True,  True],
         [ True, False],
         [False,  True],
         [False, False],
         [ True,  True]],

        [[ True,  True],
         [ True,  True],
         [False,  True],
         [ True,  True],
         [False, False],
         [False, False],
         [False,  True],
         [False,  True]],

        [[ True,  True],
         [ True,  True],
         [False, False],
         [ True,  True],
         [False, False],
         [False,  True],
         [False, False],
         [False, False]],

        [[ True,  True],
         [ True,  True],
         [ True,  True],
         [ True,  True],
         [ True,  True],
         [ True, False],
         [ True, False],
         [ True, False]]])

In [45]:
xbow2.shape

torch.Size([4, 8, 2])

In [46]:
out.shape

torch.Size([4, 8, 2])

In [47]:
mask = torch.tensor([
    [1, 0, 0],
    [1, 1, 0],
    [1, 1, 1]
])

In [48]:
x = torch.ones(3, 5, dtype=mask.dtype)

In [49]:
mask @ x

tensor([[1, 1, 1, 1, 1],
        [2, 2, 2, 2, 2],
        [3, 3, 3, 3, 3]])

In [50]:
# version 4: self-attention!
torch.manual_seed(1337)
B, T, C = 4, 8, 32 # batch, time, channels
x = torch.randn(B, T, C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)   # (16, 32)
query = nn.Linear(C, head_size, bias=False) # (16, 32)
value = nn.Linear(C, head_size, bias=False) # (16, 32)
k = key(x)      # (B, T, 16)
q = query(x)    # (B, T, 16)
wei = q @ k.transpose(-2, -1)   # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill_(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ x

In [51]:
T, C = 8, 32

In [128]:
class RNNScratch(nn.Module):
    def __init__(self, num_inputs, num_hiddens, sigma):
        super().__init__()
        self.num_inputs = num_inputs
        self.num_hiddens = num_hiddens
        self.W_xh = nn.Linear(num_inputs, num_hiddens, bias=False)
        self.W_hh = nn.Linear(num_hiddens, num_hiddens)

    def forward(self, inputs, state):
        if state is None:
            state = torch.zeros(inputs.shape[0], self.num_hiddens).to(inputs.device)
        outputs = []
        inputs = inputs.transpose(0, 1) # T, B, C
        for x in inputs:
            state = torch.tanh(self.W_xh(x) + self.W_hh(state))
            outputs.append(state)
        outputs = torch.stack(outputs).transpose(0, 1)
        return outputs, state # B, T, C
    
class RNNLMScratch(nn.Module):
    def __init__(self, rnn, num_hiddens, vocab_size, sigma):
        super().__init__()
        self.rnn = rnn
        self.num_hiddens = num_hiddens
        self.vocab_size = vocab_size
        self.W_hq = nn.Linear(num_hiddens, vocab_size)

    def clip_gradient(self):
        params = [p for p in self.parameters() if p.requires_grad]
        norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in params))
        if norm > 1:
            for p in params:
                p.grad[:] /= norm

    def forward(self, inputs, state):
        inputs = F.one_hot(inputs, self.vocab_size).float().cuda()
        outputs, state = self.rnn(inputs, state)
        outputs = self.W_hq(outputs)
        return outputs, state
    
    def generate(self, prefix, state, num_tokens):
        inputs = encode(prefix)
        out_tokens = []
        with torch.no_grad():
            for _ in range(num_tokens):
                outputs, state = self(torch.tensor([inputs]), state)
                probs = F.softmax(outputs[-1][0], 0)
                idx_next = torch.multinomial(probs, 1).item()
                out_tokens.append(idx_next)
                inputs = [idx_next]
        return prefix + decode(out_tokens)

rnn = RNNScratch(num_inputs=vocab_size, num_hiddens=128, sigma=0.01).cuda()
lm = RNNLMScratch(rnn=rnn, num_hiddens=128, vocab_size=vocab_size, sigma=0.01).cuda()

In [85]:
lm(xb, None)[0]

torch.Size([4, 8, 128])
torch.Size([4, 8, 53])


tensor([[[ 0.0861, -0.0102,  0.0835,  ...,  0.0475,  0.0476, -0.0368],
         [ 0.0862,  0.0008,  0.0278,  ..., -0.0463,  0.0247,  0.0479],
         [ 0.0978, -0.0026,  0.1069,  ..., -0.0487,  0.0127, -0.1099],
         ...,
         [ 0.0259, -0.0258,  0.0815,  ...,  0.0541,  0.0406, -0.1193],
         [ 0.0806,  0.0783,  0.0116,  ...,  0.0705,  0.0121, -0.0924],
         [ 0.0292, -0.0608,  0.0556,  ..., -0.0781,  0.0549, -0.1176]],

        [[-0.0133,  0.1349,  0.0172,  ..., -0.0185,  0.0261, -0.0100],
         [ 0.0436,  0.0236,  0.1448,  ..., -0.0646,  0.0121, -0.1283],
         [ 0.0652,  0.0138,  0.0469,  ...,  0.0631,  0.0290, -0.0763],
         ...,
         [ 0.0797,  0.0618,  0.0874,  ...,  0.0809, -0.0819, -0.1196],
         [ 0.1237,  0.0362,  0.0913,  ..., -0.0503,  0.0563, -0.1201],
         [-0.0472,  0.0312,  0.0072,  ...,  0.0288, -0.1399, -0.1144]],

        [[ 0.0710,  0.0707,  0.1259,  ...,  0.0134,  0.0146, -0.0890],
         [-0.0222,  0.1513,  0.0629,  ...,  0

In [81]:
yb.shape

torch.Size([4, 8])

In [60]:
xb, yb = get_batch('train')

In [183]:
block_size_2 = 8 # what is the maximum context length for prediction
import random
def get_example(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    i = random.randint(0, len(data) - block_size_2)
    x = data[i: i + block_size_2]
    y = data[i + 1: i + block_size_2 + 1]
    return x, y
block_size = 32
batch_size = 128
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i: i + block_size] for i in ix])
    y = torch.stack([data[i + 1: i + block_size + 1] for i in ix])
    return x, y

In [132]:
optimizer = torch.optim.AdamW(lm.parameters(), lr=1e-6)
for i in range(100000):
    xb, yb = get_batch('train')
    xb, yb = xb.to('cuda'), yb.to('cuda')
    outputs, _ = lm(xb, None)
    loss = F.cross_entropy(outputs.view(-1, 53), yb.view(-1))
    optimizer.zero_grad()
    loss.backward()
    lm.clip_gradient()
    optimizer.step()
    if (i + 1) % 1000 == 0:
        print(loss.item())
        print(lm.generate('Hi', None, 30))

2.026851177215576
Hiat myenol sheans thee ves bor 
2.0782968997955322
Hielo us nots the pendobd lem gw
2.119833469390869
Hiat tisipere Thal stithme that 
2.0527756214141846
HiEkll been And anctr aT But thi
2.109062433242798
Hie commy ture oss fretire serce
2.182922840118408
Hie Ae and bot theed the bery fo
2.0823521614074707
HiTh trineilon Kp bed he have me
2.098134756088257
Hies he nomm Botse one Tht eacce
2.076007843017578
HiCldancondseld buit ot an  sori
2.1662466526031494
Hibt ous soy whet I lardd Whough
2.1060211658477783
Hieprpver to muth enther Sorkeng
2.21968412399292
Hibagn teld Wellld fakenount you
2.1321513652801514
HiAn Aijuthin es you the wall se
2.0301949977874756
Hies ghe wist Hordey Totrourd sa
2.1435294151306152
Hi gert And gosemr therer premy 
2.0560929775238037
Hias And thewd her busteat ruse 
2.1624057292938232
Hie Medsst sake bearoo this the 
2.1275126934051514
Hief to hat dlled aik lod fforea
2.091479778289795
Hiuchedave by hixe byour leane c
2.032984733581543
Hia

In [166]:
print(lm.generate('follow', None, 30))

followound and asp awnelrt IIO Th th


In [96]:
outputs, _ = lm(embs, None)

tensor(3.9699, grad_fn=<NllLossBackward0>)

In [48]:
x, y = get_example('train')

In [69]:
embs = F.one_hot(x, vocab_size).float()

In [ ]:
class LSTMScratch(nn.Module):
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()
        self.num_inputs = num_inputs
        self.num_hiddens = num_hiddens
        # Forget gate (how much of the long-term memory is retained)
        self.W_xf = nn.Parameter(torch.randn(num_inputs, num_hiddens) * sigma)
        self.W_hf = nn.Parameter(torch.randn(num_hiddens, num_hiddens) * sigma)
        self.b_f = nn.Parameter(torch.randn(num_hiddens))
        # Input gate
        self.W_xi = nn.Parameter(torch.randn(num_inputs, num_hiddens) * sigma)
        self.W_hi = nn.Parameter(torch.randn(num_hiddens, num_hiddens) * sigma)
        self.b_i = nn.Parameter(torch.randn(num_hiddens))
        # Input node
        self.W_xc = nn.Parameter(torch.randn(num_inputs, num_hiddens) * sigma)
        self.W_hc = nn.Parameter(torch.randn(num_hiddens, num_hiddens) * sigma)
        self.b_c = nn.Parameter(torch.randn(num_hiddens))
        # Output gate
        self.W_xo = nn.Parameter(torch.randn(num_inputs, num_hiddens) * sigma)
        self.W_ho = nn.Parameter(torch.randn(num_hiddens, num_hiddens) * sigma)
        self.b_o = nn.Parameter(torch.randn(num_hiddens))

    def forward(self, inputs, H_C):
        # inputs -> (B, N, num_inputs), H_C -> (B, N, num_hiddens), (B, N, num_hiddens)
        if H_C is None:
            H = torch.zeros(inputs.shape[0], self.num_hiddens).to(inputs.device) # (B, num_hiddens)
            C = torch.zeros(inputs.shape[0], self.num_hiddens).to(inputs.device) # (B, num_hiddens)
        else:
            H, C = H_C

        inputs = inputs.transpose(0, 1) # (N, B, num_hiddens)
        outputs = []
        for x in inputs:
            f = F.sigmoid(x @ self.W_xf + H @ self.W_hf + self.b_f) # Forget some of the long-term memory (C), which is decided by the short-term input (H, x)
            i = F.sigmoid(x @ self.W_xi + H @ self.W_hi + self.b_i) # Decide how much of short-term memory (c) to be incorporated in to long-term memory (C),
                                                                    # decided by the short-term input (H, x)
            c = F.tanh(x @ self.W_xc + H @ self.W_hc + self.b_c)    # Calculate the short-term memory to be incorporated (c) in to the long-term memory (C)
                                                                    # decided by the short-term input (H, x)
            C = C * f + i * c   # Update memory cell internal state, i.e. calculate the new long-term memory (C)
            o = F.sigmoid(x @ self.W_xo + H @ self.W_ho + self.b_o) # Calculate how much of the new long-term memory (C) will be come the new short-term memory (H)
            H = F.tanh(C) * o   # Calculate the new short-term memory, or output hidden state (H)
            outputs.append(H)
        outputs = torch.stack(outputs).transpose(0, 1)
        return outputs, (H, C)
    
lstm = LSTMScratch(num_inputs=vocab_size, num_hiddens=128, sigma=0.01).cuda()
lm = RNNLMScratch(rnn=lstm, num_hiddens=128, vocab_size=vocab_size, sigma=0.01).cuda()

In [190]:
optimizer = torch.optim.AdamW(lm.parameters(), lr=1e-3)
for i in range(100000):
    xb, yb = get_batch('train')
    xb, yb = xb.to('cuda'), yb.to('cuda')
    outputs, _ = lm(xb, None)
    loss = F.cross_entropy(outputs.view(-1, 53), yb.view(-1))
    optimizer.zero_grad()
    loss.backward()
    # lm.clip_gradient()
    optimizer.step()
    if (i + 1) % 1000 == 0:
        print(loss.item())
        print(lm.generate('Hi', None, 30))

1.9805755615234375
Hi sust Mersuant me I wigceithit
1.796811819076538
Hi carr atperst on our mencous t
1.6928287744522095
Hi mill the wands thee the guder
1.6483670473098755
Hiins forwich Hell me For Ill th
1.6404027938842773
HiA parrint with my tands the fo
1.59148108959198
HiAs I boy By daint I am of the 
1.5192532539367676
Hie so have with a cralisuin tod
1.5318965911865234
Hiesely cannot fall quire  Butti
1.510345220565796
Hi should be remever Mikes  ISAB
1.518970012664795
Hiak if I dear in by as will but
1.4707436561584473
HiA Thou art devedle my than her
1.4480549097061157
Hier a joy O hear me princed god
1.4776484966278076
Hie  GRELIUS Why my love God for
1.462446928024292
Hie consent that stay he consent
1.4485869407653809
Hi would against thou thought of
1.447938323020935
Hielons God Why then I cument  L
1.4610235691070557
Hie down a duke now comaling Wit
1.4112663269042969
Hie away  Clwan Appearing Say I 
1.3954112529754639
Hiowell on his liferey go qualie
1.4077353477478027


In [ ]:
optimizer = torch.optim.AdamW(lm.parameters(), lr=1e-4)
for i in range(100000):
    xb, yb = get_batch('train')
    xb, yb = xb.to('cuda'), yb.to('cuda')
    outputs, _ = lm(xb, None)
    loss = F.cross_entropy(outputs.view(-1, 53), yb.view(-1))
    optimizer.zero_grad()
    loss.backward()
    # lm.clip_gradient()
    optimizer.step()
    if (i + 1) % 1000 == 0:
        print(loss.item())
        print(lm.generate('Jesus', None, 30))

In [214]:
lm.generate('as', None, 30)

'ast the girld infory of you  CAP'

In [ ]:
# version 4: self-attention!
torch.manual_seed(1337)
B, T, C = 4, 8, 32 # batch, time, channels
x = torch.randn(B, T, C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)   # (16, 32)
query = nn.Linear(C, head_size, bias=False) # (16, 32)
value = nn.Linear(C, head_size, bias=False) # (16, 32)
k = key(x)      # (B, T, 16)
q = query(x)    # (B, T, 16)
wei = q @ k.transpose(-2, -1)   # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill_(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v

In [ ]:
block_size = 32
batch_size = 128
with open('input.txt', 'r') as f:
    text = f.read()
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars)}
itos = { i:ch for i,ch in enumerate(chars)}
for ch in chars:
    assert itos[stoi[ch]] == ch
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
assert decode(encode(text[100-130])) == text[100-130]
data = torch.tensor(encode(text))

# Split train and test data
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

train_data[:block_size + 1]

x = train_data[:block_size]
y = train_data[1:block_size + 1]

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i: i + block_size] for i in ix])
    y = torch.stack([data[i + 1: i + block_size + 1] for i in ix])
    return x, y

class Head(nn.Module):
    """one-head of self-attention"""
    
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x) # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        wei = q @ k.transpose(-2, -1) # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        wei = wei.masked_fill_(self.tril[:T, :T] == 0, float('-inf')) 
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)   # (B, T, head_size)
        out = wei @ v   # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out
    

class AttentionLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, head_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.sa_head = Head(n_embd, head_size)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        x = self.token_embedding(idx)   # (B, T, n_embd)
        x = self.sa_head(x)    # (B, T, n_embd)
        logits = self.lm_head(x)    # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # become (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T + 1)
        return idx
    
lm = AttentionLanguageModel(vocab_size, n_embd=32, head_size=32)

In [56]:
optimizer = torch.optim.Adam(lm.parameters(), lr=1e-3)
for r in range(100000):
    xb, yb = get_batch('train')
    logits, loss = lm(xb, yb)
    print(r, loss.item())
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

0 4.211729049682617
1 4.198624134063721
2 4.183403968811035
3 4.171236515045166
4 4.162321090698242
5 4.148618221282959
6 4.135714054107666
7 4.12050724029541
8 4.115090847015381
9 4.100780487060547
10 4.088863849639893
11 4.070330619812012
12 4.053954601287842
13 4.041201114654541
14 4.0269389152526855
15 4.020029544830322
16 3.9982244968414307
17 3.9831619262695312
18 3.9670705795288086
19 3.9493796825408936
20 3.929892063140869
21 3.923649549484253
22 3.885680913925171
23 3.8743646144866943
24 3.859391212463379
25 3.851841449737549
26 3.8031351566314697
27 3.7912521362304688
28 3.78946590423584
29 3.744887590408325
30 3.7236998081207275
31 3.7286479473114014
32 3.70849609375
33 3.6523325443267822
34 3.6505520343780518
35 3.626300811767578
36 3.573739767074585
37 3.5913827419281006
38 3.5657851696014404
39 3.557241201400757
40 3.528892755508423
41 3.5184061527252197
42 3.483858823776245
43 3.4703755378723145
44 3.4267313480377197
45 3.4240992069244385
46 3.395564556121826
47 3.408941

In [57]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(lm.generate(context, max_new_tokens=2000)[0].tolist()))


T:
Whimuromy on tme at dlakn ntofurss hourenfueend ntourt tor;
remather sp t her me totisth
fifa he fous fofar w,
UCKw!
DKIENRINII s
T, RI:
RYou lkin t ts, wo en feron charequisond whyorw terd thickior ofan p,
Thesthathyo.
KIthond O:
Sorme t thathethite hindy follour bin tiser,
Sotolatre

Ofantofidrellor mr ur dore LOnoocoff me
AWARENGo be.
S oo bly lothat hy rom gher? erofrman.


TIname he ' tledintooldstears ifrg ay tod by yowopp astatorsianocas be,
Tuade t as?
RIDUCLY:
Duryulisu whra ste IO kextant, ouperiffin me
OFr f ht
Ans wig?
RPOMEOR:
Tholdreve main bovoss ldereast himouendve y
Tha'sedaled alches acunour allfe shoupill'strethe ad; Has s, t out chotho t urcothet asant as ly?
RE hus?

BLave wke: lichat Gorl hesemes htos lo ly idsouthancccteessk! litidse the det t fot s p.Se y meapee Poo; a k'ss tpearede whtheded agruitis hertero.


Yeravelu otive win thy, cesha'enr.


And, Cruches lo avyoons his, att s sotonou woroun or ld ghe
drtry by ospe ghale ththe croth vou witses a engbe i

In [ ]:
eval_iters = 200

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """one-head of self-attention"""
    
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x) # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        wei = q @ k.transpose(-2, -1) # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        wei = wei.masked_fill_(self.tril[:T, :T] == 0, float('-inf')) 
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)   # (B, T, head_size)
        out = wei @ v   # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out
    
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.attn = nn.ModuleList([Head(n_embd, head_size) for _ in range(n_embd//head_size)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x): # (B, T, n_embd)
        out = torch.concat([head(x) for head in self.attn], dim=-1)
        out = self.proj(out)
        return out
    
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity"""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )
    def forward(self, x):
        return self.net(x)
    
class Block(nn.Module):
    """ Transformer block: communication followed by computation"""
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, head_size)
        self.ln_sa = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)
        self.ln_ffwd = nn.LayerNorm(n_embd)

    def forward(self, x):
        # # Original transformer paper
        # x = self.ln_sa(x + self.sa(x))
        # x = self.ln_ffwd(x + self.ffwd(x))
        # Later improvement
        x = x + self.sa(self.ln_sa(x)) 
        x = x + self.ffwd(self.ln_ffwd(x))
        return x

class MultiHeadAttentionLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, head_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.multi_head_attention = MultiHeadAttention(n_embd, head_size)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        x = self.token_embedding(idx)   # (B, T, n_embd)
        x = self.multi_head_attention(x)    # (B, T, n_embd)
        logits = self.lm_head(x)    # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # become (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T + 1)
        return idx
    
lm = MultiHeadAttentionLanguageModel(vocab_size, 32, 4)

In [75]:
optimizer = torch.optim.Adam(lm.parameters(), lr=1e-3)
for r in range(100000):
    xb, yb = get_batch('train')
    logits, loss = lm(xb, yb)
    print(r, loss.item())
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

torch.Size([128, 32, 32])
0 4.211226463317871
torch.Size([128, 32, 32])
1 4.2025275230407715
torch.Size([128, 32, 32])
2 4.193061351776123
torch.Size([128, 32, 32])
3 4.181256294250488
torch.Size([128, 32, 32])
4 4.169564723968506
torch.Size([128, 32, 32])
5 4.155519962310791
torch.Size([128, 32, 32])
6 4.150667190551758
torch.Size([128, 32, 32])
7 4.139218330383301
torch.Size([128, 32, 32])
8 4.120359897613525
torch.Size([128, 32, 32])
9 4.107284069061279
torch.Size([128, 32, 32])
10 4.104861736297607
torch.Size([128, 32, 32])
11 4.095818042755127
torch.Size([128, 32, 32])
12 4.081432819366455
torch.Size([128, 32, 32])
13 4.063830852508545
torch.Size([128, 32, 32])
14 4.067719459533691
torch.Size([128, 32, 32])
15 4.043082237243652
torch.Size([128, 32, 32])
16 4.02440881729126
torch.Size([128, 32, 32])
17 4.000078201293945
torch.Size([128, 32, 32])
18 4.007418632507324
torch.Size([128, 32, 32])
19 3.9837260246276855
torch.Size([128, 32, 32])
20 3.9569931030273438
torch.Size([128, 32, 

KeyboardInterrupt: 

In [78]:
torch.arange(10)

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])